Finding good hyperparameters is one of the most time-consuming parts of applied machine learning. Scikit-learn provides `GridSearchCV` and `RandomizedSearchCV` to automate hyperparameter search with built-in cross-validation. The `n_jobs` parameter lets you parallelize this work across multiple CPU cores, which can dramatically reduce wall-clock time.

Since the Iris dataset from earlier notebooks has only 150 rows, any hyperparameter search finishes almost instantly regardless of how many cores you use. To see meaningful timing differences, this notebook generates a larger dataset using scikit-learn's `make_classification`.

## Import Libraries

In [ ]:
%matplotlib inline

import time
import os

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

## Create a Larger Dataset

Scikit-learn's `make_classification` generates a synthetic classification dataset. We create 20,000 samples with 20 features — over 130 times larger than Iris — so that each model fit takes enough time for the parallelism benefits to be visible.

In [ ]:
X, y = make_classification(
    n_samples=20_000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=0
)

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')

## Splitting Data into Training and Test Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

## Baseline: A Single Decision Tree

Before tuning hyperparameters, let's see how a single decision tree with default settings performs on this dataset.

In [ ]:
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X_train, y_train)

print(f'Training accuracy: {clf.score(X_train, y_train):.4f}')
print(f'Test accuracy:     {clf.score(X_test, y_test):.4f}')

The default tree overfits: it memorizes the training data perfectly but does worse on the test set. Tuning hyperparameters like `max_depth` and `min_samples_leaf` can reduce this overfitting.

## Hyperparameter Tuning with GridSearchCV

In the Decision Trees notebook, we searched over `max_depth` values using a manual for loop. `GridSearchCV` automates this and adds cross-validation, which gives a more reliable estimate of how well each hyperparameter setting will generalize.

When we search over multiple hyperparameters at once, the number of combinations grows quickly.

In [ ]:
param_grid = {
    'max_depth': [4, 8, 12, 16, 20, None],
    'min_samples_split': [2, 10, 50],
    'min_samples_leaf': [1, 4, 16],
    'criterion': ['gini', 'entropy']
}

total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)

print(f'Hyperparameter combinations: {total_combinations}')
print(f'With 5-fold cross-validation: {total_combinations * 5} total model fits')

### Single Core (n_jobs=1)

By default, `GridSearchCV` uses a single CPU core. All 540 fits run one after another.

In [ ]:
start = time.time()

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=1
)
grid_search.fit(X_train, y_train)

time_grid_1 = time.time() - start

print(f'Best CV accuracy:  {grid_search.best_score_:.4f}')
print(f'Test accuracy:     {grid_search.score(X_test, y_test):.4f}')
print(f'Best parameters:   {grid_search.best_params_}')
print(f'\nTime (1 core): {time_grid_1:.1f} seconds')

### All Cores (n_jobs=-1)

Setting `n_jobs=-1` tells scikit-learn to use all available CPU cores. Each hyperparameter–fold combination is independent, so they can run in parallel.

In [ ]:
print(f'CPU cores available: {os.cpu_count()}')

start = time.time()

grid_search_parallel = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search_parallel.fit(X_train, y_train)

time_grid_all = time.time() - start

print(f'\nBest CV accuracy:  {grid_search_parallel.best_score_:.4f}')
print(f'Test accuracy:     {grid_search_parallel.score(X_test, y_test):.4f}')
print(f'Best parameters:   {grid_search_parallel.best_params_}')
print(f'\nTime ({os.cpu_count()} cores): {time_grid_all:.1f} seconds')
print(f'Speedup: {time_grid_1 / time_grid_all:.1f}x faster')

## RandomizedSearchCV

When the search space is very large, trying every combination becomes impractical. `RandomizedSearchCV` samples a fixed number of candidates from the parameter space. This often finds comparably good hyperparameters in much less time.

In [ ]:
param_distributions = {
    'max_depth': list(range(2, 30)) + [None],
    'min_samples_split': list(range(2, 100)),
    'min_samples_leaf': list(range(1, 50)),
    'criterion': ['gini', 'entropy']
}

start = time.time()

random_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_distributions,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=0
)
random_search.fit(X_train, y_train)

time_random = time.time() - start

print(f'Candidates sampled: 50 (from a much larger space)')
print(f'Best CV accuracy:   {random_search.best_score_:.4f}')
print(f'Test accuracy:      {random_search.score(X_test, y_test):.4f}')
print(f'Best parameters:    {random_search.best_params_}')
print(f'\nTime: {time_random:.1f} seconds')

## Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

methods = [
    f'GridSearchCV\n(1 core)',
    f'GridSearchCV\n({os.cpu_count()} cores)',
    f'RandomizedSearchCV\n({os.cpu_count()} cores)'
]
times = [time_grid_1, time_grid_all, time_random]
colors = ['#c44e52', '#4c72b0', '#55a868']

bars = ax.bar(methods, times, color=colors, width=0.5, edgecolor='black', linewidth=0.8)

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(times) * 0.02,
            f'{t:.1f}s', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Time (seconds)', fontsize=16)
ax.set_title('Hyperparameter Search: Timing Comparison', fontsize=18)
ax.tick_params(labelsize=14)
ax.grid(True, axis='y', zorder=0, linestyle=':', color='k')
ax.set_axisbelow(True)
fig.tight_layout()

## When Does Additional Compute Help?

Parallelism helps the most when you have many independent tasks to run. Grid search and random search are natural fits because each hyperparameter–fold combination can be evaluated on its own.

Things to keep in mind:

- **More cores help most** when each individual model fit takes meaningful time. On very small datasets like Iris (150 rows), the overhead of distributing work across cores can actually exceed the time saved.
- **RandomizedSearchCV** is often the better strategy for large search spaces. It explores more of the space per unit of time and frequently finds comparably good results.
- **Hardware matters**: the speedup you see scales with the number of cores available. A laptop with 4 cores will see a smaller speedup than a workstation with 16 or more.

For more on this topic, see [How to Speed up Scikit-Learn Model Training](https://medium.com/@GalarnykMichael/how-to-speed-up-scikit-learn-model-training-aaf17e2d1e1).